# Learned-play sweep on Colab (Amendment 2, B4)

Runs **one sweep run per Colab session** (free tier: 2 CPU cores, about 4-7 h per
run, sessions stop after about 12 h). The registry (datasets, runs) lives on
Google Drive, so every session adds to the same registry; feature caches go to
the session's local disk and are rebuilt each session.

Before the first session:
1. Add a Colab secret `GITHUB_TOKEN` (key icon on the left): a GitHub token that can
   read the repo cell 2 clones (edit its URL for your own fork).
2. Upload `results/colab-registry.zip` (made on the laptop) to
   `MyDrive/sts-for-ai/colab-registry.zip`. Cell 1 unpacks it into the registry the
   first time.

Each session: Runtime > Run all, after setting `RUN` in the run cell. Keep the tab
open and visible; an idle tab can be disconnected. Order: `score-s0` first (trains
the score round-0 net), then `hp-s0` (trains the HP one); every other run reuses
its target's round-0 net from the registry. When all 8 are done, run the last cell
for the table.

## 1. Drive and the registry

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
REG = '/content/drive/MyDrive/sts-for-ai/registry'
ZIP = '/content/drive/MyDrive/sts-for-ai/colab-registry.zip'
os.environ['REG'] = REG
os.environ['STS_CACHE_DIR'] = '/content/cache'  # local disk; Drive holds datasets and runs
os.makedirs(f'{REG}/logs', exist_ok=True)
POOLS = ['799a659bb7ac', 'b3614c9f06db', '320be46d488d', 'ec8ed760895b']  # score / HP pool, score / HP validation
missing = [d for d in POOLS if not os.path.exists(f'{REG}/datasets/{d}/manifest.json')]
if missing and os.path.exists(ZIP):
    zipfile.ZipFile(ZIP).extractall(REG)
    missing = [d for d in POOLS if not os.path.exists(f'{REG}/datasets/{d}/manifest.json')]
assert not missing, f'datasets missing from {REG}/datasets: {missing}; upload colab-registry.zip to {ZIP}'
print('registry ok:', sorted(os.listdir(f'{REG}/datasets')))

## 2. Code (token from the Colab secret)

In [ ]:
import subprocess
from google.colab import userdata
REPO = '/content/sts-for-ai'
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/Mainzu/sts-for-ai-private.git"
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--quiet', '--branch', 'main', url, REPO], check=True)
    # Keep the token out of the clone's saved config.
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin',
                    'https://github.com/Mainzu/sts-for-ai-private.git'], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--quiet', url, 'main'], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

## 3. Build the engine module (Rust) and install Python packages (~10 min)

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!pip install -q numpy maturin
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q ./py

## 4. Quick checks: bindings, and stored games replay identically here

In [ ]:
!python py/tests/test_sts.py
import sys
sys.path.insert(0, f'{REPO}/scripts')
import registry, sts
VAL = '320be46d488d'
m = registry.load_manifest(VAL, root=REG)
assert m['rules_version'] == registry.rules_version(), (
    f"rules version differs: dataset {m['rules_version']} vs engine {registry.rules_version()}; "
    'the clone is not the commit the datasets were made with')
for g in registry.load_games(VAL, root=REG)[:20]:
    c = sts.Combat(g['deck'], g['encounter'], seed=g['seed'], hp=g['hp'], character=g['character'],
                   no_death=bool(g.get('no_death')))
    for a in g['actions']:
        c.step(registry.decode_action(a))
    assert c.is_over() and abs(c.score() - g['outcome']['score']) < 1e-9, 'a stored game replays differently here'
print('20 stored games replay identically')

## 5. Speed here, and each run's projected hours

In [ ]:
import time
import numpy as np
import learn, selfplay, stsnet, sweep
print('CPU cores:', os.cpu_count())
specs, _ = selfplay.make_specs(34, 7, selfplay.train_deck)  # 102 games
t = time.perf_counter()
sts.run('search', specs, **learn.SELFPLAY_SEARCH)
heur = len(specs) / (time.perf_counter() - t)
f = selfplay.features('effects')
d = f.dims
norm = stsnet.Normalizer({k: (np.zeros(d[dk]), np.ones(d[dk]))
                          for k, dk in (('player', 'player'), ('monsters', 'monster'), ('cards', 'card'), ('rows', 'row'))})
meta = {'card_repr': 'effects', **{f'known_{k}': v for k, v in f.vocab.items()}}
ev = stsnet.rust_evaluator(stsnet.ValueNet(d), norm, meta)  # untrained: as fast as a trained net; Rust inference as the sweep uses
t = time.perf_counter()
mixed = specs[::2]  # every encounter group (make_specs lists them in order; hallway fights are short)
sts.run('search', mixed, evaluator=ev, **learn.SELFPLAY_SEARCH)
net = len(mixed) / (time.perf_counter() - t)
print(f'self-play here: heuristic {heur:.0f} games/s, net {net:.2f} games/s '
      f"(laptop: {sweep.RATES[('heuristic', False)]:.0f} and {sweep.RATES[('net', False)]:.0f})")
# An untrained net plays differently, so trust whichever ratio is slower.
slow = max(sweep.RATES[('net', False)] / net, sweep.RATES[('heuristic', False)] / heur)
for r in sweep.projection()[0]:
    print(f"{r['name']:<16} ~{r['minutes'] * slow / 60:.1f} h here")

## 6. One run (one per session)

In [ ]:
RUN = 'score-s0'  #@param ['score-s0', 'score+branch-s0', 'hp-s0', 'hp+branch-s0', 'score-s1', 'score+branch-s1', 'hp-s1', 'hp+branch-s1']
!python -u scripts/sweep.py --only {RUN} --root "{REG}" 2>&1 | tee -a "{REG}/logs/{RUN}.log"

## 7. The table (after all runs)

Writes `runs/sweep-<date>.md` in the registry from the newest finished run of each
plan row (missing rows are listed). Bring `registry/runs/index.csv`'s new lines and
the table back into the repo's `runs/` (download them, or copy from Drive).

In [ ]:
!python scripts/sweep.py --table --root "{REG}"